In [1]:
from __future__ import print_function, division
import sys
import os
import rhalphalib as rl
import numpy as np
import scipy.stats
import pickle
import ROOT
rl.util.install_roofit_helpers()
rl.ParametericSample.PreferRooParametricHist = False

Welcome to JupyROOT 6.18/00


In [2]:
def expo_sample(norm, scale, obs):
    cdf = scipy.stats.expon.cdf(scale=scale, x=obs.binning) * norm
    return (np.diff(cdf), obs.binning, obs.name)

In [3]:
model = rl.Model("testMonojet")
# lumi = rl.NuisanceParameter('CMS_lumi', 'lnN')
jec = rl.NuisanceParameter('CMS_jec', 'shape')
ele_id_eff = rl.NuisanceParameter('CMS_ele_id_eff', 'shape')
pho_id_eff = rl.NuisanceParameter('CMS_pho_id_eff', 'shape')
gamma_to_z_ewk = rl.NuisanceParameter('Theory_gamma_z_ewk', 'shape')

recoilbins = np.linspace(300, 1200, 13)
recoil = rl.Observable('recoil', recoilbins)

In [4]:

gammaCh = rl.Channel("gammaCh")
model.addChannel(gammaCh)

gammaTemplate = expo_sample(2000, 450, recoil)
gammaJetsMC = rl.TemplateSample('gammaJetsMC', rl.Sample.BACKGROUND, gammaTemplate)
gammaJetsMC.setParamEffect(jec, np.random.normal(loc=1, scale=0.05, size=recoil.nbins))
gammaJetsMC.setParamEffect(pho_id_eff, np.random.normal(loc=1, scale=0.02, size=recoil.nbins))


gammaCh.setObservation(expo_sample(2000, 450, recoil))

In [19]:
workspace = ROOT.RooWorkspace(model.name)

In [39]:
!pwd

/cms/scratch/runiyal/decaf_aug20/decaf/analysis/Rishabh_NB


In [40]:
outputPath = '/cms/scratch/runiyal/decaf_aug20/decaf/analysis/Rishabh_NB'

In [42]:
workspace = ROOT.RooWorkspace(model.name)
model.renderRoofit(workspace)
workspace.writeToFile(os.path.join(outputPath, "%s.root" % model.name))

False

In [48]:
import ROOT